In [1]:
!pip install gensim

In [2]:
import os
import numpy as np
import torch
import pandas as pd
from tqdm import tqdm
from scipy import sparse
from google.colab import drive
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
import sys
import wandb
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
import re
import scipy
import json
import gensim.downloader
import os
import numpy as np
import torch
import pandas as pd
import gensim.downloader
from tqdm import tqdm
from scipy import sparse
from google.colab import drive
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
import torch.nn as nn
from sklearn.metrics import f1_score
from gensim.models import Word2Vec
from torch.utils.data import Dataset, DataLoader
from collections import Counter

In [3]:
class TextDataset(Dataset):
    def __init__(self, X, y):
        if scipy.sparse.issparse(X):
            self.X = X.tocsr()
        else:
            self.X = np.asarray(X)
        self.y = np.asarray(y)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        x_item = self.X[idx]
        if scipy.sparse.issparse(x_item):
            x_item = x_item.toarray().squeeze()

        return (
            torch.tensor(x_item, dtype=torch.float32),
            torch.tensor(self.y[idx], dtype=torch.float32)
        )

In [4]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()

    total_loss = 0.0
    total_samples = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()

        logits = model(x)
        loss = criterion(logits, y.to(device, dtype=torch.float32))

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)
        total_samples += x.size(0)

    return total_loss / total_samples

@torch.no_grad()

def evaluate(model, loader, criterion, device, threshold=0.5):
    model.eval()

    total_loss = 0.0
    all_preds = []
    all_targets = []

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        logits = model(x)
        loss = criterion(logits, y)

        probs = torch.sigmoid(logits)
        preds = (probs > threshold).cpu().numpy()

        all_preds.append(preds)
        all_targets.append(y.cpu().numpy())

        total_loss += loss.item() * x.size(0)

    all_preds = np.vstack(all_preds)
    all_targets = np.vstack(all_targets)

    micro_f1 = f1_score(all_targets, all_preds, average="micro")

    return total_loss / len(loader.dataset), micro_f1

def run_experiment(train_loader, test_loader, model, experiment_name):

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Running experiment on device: {device}")

    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


    y_train_np = np.asarray(y_train)
    pos_counts = y_train_np.sum(axis=0)
    neg_counts = len(y_train_np) - pos_counts
    pos_weight = torch.tensor(neg_counts / (pos_counts + 1e-6), dtype=torch.float32).to(device)
    criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    wandb.init(
      project="Genre_Detector",
      name=experiment_name,
    )

    for epoch in range(10):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_f1 = evaluate(model, test_loader, criterion, device, threshold=0.5)

        print(f"Epoch {epoch:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val micro-F1: {val_f1:.4f}")

        wandb.log({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_micro_f1": val_f1
        })

    wandb.finish()

In [5]:
sys.path.append("/content/drive/MyDrive/Genre_Detector")
drive.mount('/content/drive')
df = pd.read_csv("/content/drive/MyDrive/data_goodreads.csv")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
def parse_genres(x):
    if not isinstance(x, str):
        return []
    return re.findall(r'"(.*?)"', x)

def encode_genres(genres):
  vec = np.zeros(30, dtype=np.float32)
  for g in genres:
      if g in genre_to_idx:
          vec[genre_to_idx[g]] = 1.0
  return vec

df["genres"] = df["genres"].apply(parse_genres)
top30 = df["genres"].explode().value_counts().head(30)
top30 = df["genres"].explode().value_counts().head(30).index.tolist()
genre_to_idx = {g: i for i, g in enumerate(top30)}
y = np.stack(df["genres"].apply(encode_genres))

In [7]:
X = df["summary"]

#CONFIG
X = X[:100000]
y = y[:100000]

X_train, X_test, y_train, y_test = train_test_split(
  X, y, test_size=0.2, random_state=42
)


MLP

In [8]:
class MLP_basic(nn.Module):
    def __init__(self, input_dim=300, num_classes=30):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),

            nn.Linear(512, 256),
            nn.ReLU(),

            nn.Linear(256, num_classes)
          )

    def forward(self, x):
        return self.net(x)

In [9]:
class ResidualBlock(nn.Module):
  def __init__(self, dim, dropout_rate=0.3):
      super().__init__()
      self.block = nn.Sequential(
          nn.Linear(dim, dim),
          nn.BatchNorm1d(dim),
          nn.ReLU(),
          nn.Dropout(dropout_rate),
          nn.Linear(dim, dim),
          nn.BatchNorm1d(dim),
          nn.Dropout(dropout_rate)
      )
      self.relu = nn.ReLU()

  def forward(self, x):
    return self.relu(self.block(x) + x)

class ResNet_MLP(nn.Module):
    def __init__(self, input_dim=300, hidden_dim=512, num_classes=30, num_blocks=3):
        super().__init__()

        self.input_layer = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU()
        )

        self.res_blocks = nn.Sequential(
            *[ResidualBlock(hidden_dim) for _ in range(num_blocks)]
        )

        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        out = self.input_layer(x)
        out = self.res_blocks(out)
        out = self.classifier(out)
        return out


In [13]:
vectorizer = CountVectorizer(max_features=300, stop_words='english')

X_train_bow = vectorizer.fit_transform(X_train)
X_test_bow = vectorizer.transform(X_test)

train_dataset = TextDataset(X_train_bow, y_train)
test_dataset = TextDataset(X_test_bow, y_test)

train_loader_bow = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader_bow = DataLoader(test_dataset, batch_size=128, shuffle=False)

In [14]:
model = MLP_basic(
            input_dim=300,
            num_classes=30
        ).to("cuda")

run_experiment(train_loader_bow, test_loader_bow, model, "MLP_bow")

Running experiment on device: cuda


Epoch 00 | Train Loss: 0.9551 | Val Loss: 0.8958 | Val micro-F1: 0.2669
Epoch 01 | Train Loss: 0.8555 | Val Loss: 0.8741 | Val micro-F1: 0.2764
Epoch 02 | Train Loss: 0.8153 | Val Loss: 0.8696 | Val micro-F1: 0.2722
Epoch 03 | Train Loss: 0.7792 | Val Loss: 0.8732 | Val micro-F1: 0.2826
Epoch 04 | Train Loss: 0.7398 | Val Loss: 0.8982 | Val micro-F1: 0.2841
Epoch 05 | Train Loss: 0.6992 | Val Loss: 0.9331 | Val micro-F1: 0.3061
Epoch 06 | Train Loss: 0.6591 | Val Loss: 0.9832 | Val micro-F1: 0.3077
Epoch 07 | Train Loss: 0.6207 | Val Loss: 1.0420 | Val micro-F1: 0.3099
Epoch 08 | Train Loss: 0.5859 | Val Loss: 1.1486 | Val micro-F1: 0.3182
Epoch 09 | Train Loss: 0.5539 | Val Loss: 1.2315 | Val micro-F1: 0.3134


epoch,▁▂▃▃▄▅▆▆▇█
train_loss,█▆▆▅▄▄▃▂▂▁
val_loss,▂▁▁▁▂▂▃▄▆█
val_micro_f1,▁▂▂▃▃▆▇▇█▇
epoch,9
train_loss,0.55393
val_loss,1.2315
val_micro_f1,0.31338


In [15]:
model = ResNet_MLP().to("cuda")

run_experiment(train_loader_bow, test_loader_bow, model, "ResNet_bow")

Running experiment on device: cuda


Epoch 00 | Train Loss: 0.9920 | Val Loss: 0.8958 | Val micro-F1: 0.2580
Epoch 01 | Train Loss: 0.8818 | Val Loss: 0.8821 | Val micro-F1: 0.2664
Epoch 02 | Train Loss: 0.8447 | Val Loss: 0.8868 | Val micro-F1: 0.2534
Epoch 03 | Train Loss: 0.8166 | Val Loss: 0.8855 | Val micro-F1: 0.2713
Epoch 04 | Train Loss: 0.7882 | Val Loss: 0.8857 | Val micro-F1: 0.2612
Epoch 05 | Train Loss: 0.7592 | Val Loss: 0.9023 | Val micro-F1: 0.2771
Epoch 06 | Train Loss: 0.7329 | Val Loss: 0.9259 | Val micro-F1: 0.2829
Epoch 07 | Train Loss: 0.7086 | Val Loss: 0.9468 | Val micro-F1: 0.2868
Epoch 08 | Train Loss: 0.6839 | Val Loss: 0.9658 | Val micro-F1: 0.2856
Epoch 09 | Train Loss: 0.6609 | Val Loss: 0.9985 | Val micro-F1: 0.2981


epoch,▁▂▃▃▄▅▆▆▇█
train_loss,█▆▅▄▄▃▃▂▁▁
val_loss,▂▁▁▁▁▂▄▅▆█
val_micro_f1,▂▃▁▄▂▅▆▆▆█
epoch,9
train_loss,0.66091
val_loss,0.99849
val_micro_f1,0.29814


In [18]:
tfidf_vectorizer = TfidfVectorizer(max_features=300, stop_words='english')

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

train_dataset_tfidf = TextDataset(X_train_tfidf, y_train)
test_dataset_tfidf = TextDataset(X_test_tfidf, y_test)

train_loader_tfidf = DataLoader(train_dataset_tfidf, batch_size=128, shuffle=True)
test_loader_tfidf = DataLoader(test_dataset_tfidf, batch_size=128, shuffle=False)

In [19]:
model = MLP_basic(
            input_dim=300,
            num_classes=30
        ).to("cuda")

run_experiment(train_loader_tfidf, test_loader_tfidf, model, "MLP_tfidf")

Running experiment on device: cuda


Epoch 00 | Train Loss: 0.9790 | Val Loss: 0.9091 | Val micro-F1: 0.2761
Epoch 01 | Train Loss: 0.8805 | Val Loss: 0.8877 | Val micro-F1: 0.2778
Epoch 02 | Train Loss: 0.8524 | Val Loss: 0.8729 | Val micro-F1: 0.2740
Epoch 03 | Train Loss: 0.8294 | Val Loss: 0.8686 | Val micro-F1: 0.2778
Epoch 04 | Train Loss: 0.8077 | Val Loss: 0.8744 | Val micro-F1: 0.2941
Epoch 05 | Train Loss: 0.7848 | Val Loss: 0.8741 | Val micro-F1: 0.2893
Epoch 06 | Train Loss: 0.7601 | Val Loss: 0.8795 | Val micro-F1: 0.2883
Epoch 07 | Train Loss: 0.7342 | Val Loss: 0.9012 | Val micro-F1: 0.3003
Epoch 08 | Train Loss: 0.7067 | Val Loss: 0.9082 | Val micro-F1: 0.2928
Epoch 09 | Train Loss: 0.6782 | Val Loss: 0.9317 | Val micro-F1: 0.2928


epoch,▁▂▃▃▄▅▆▆▇█
train_loss,█▆▅▅▄▃▃▂▂▁
val_loss,▅▃▁▁▂▂▂▅▅█
val_micro_f1,▂▂▁▂▆▅▅█▆▆
epoch,9
train_loss,0.67818
val_loss,0.93171
val_micro_f1,0.29277


In [22]:
model = ResNet_MLP().to("cuda")

run_experiment(train_loader_tfidf, test_loader_tfidf, model, "ResNet_tfidf")

Running experiment on device: cuda


Epoch 00 | Train Loss: 0.9913 | Val Loss: 0.9065 | Val micro-F1: 0.2548
Epoch 01 | Train Loss: 0.8816 | Val Loss: 0.8969 | Val micro-F1: 0.2696
Epoch 02 | Train Loss: 0.8399 | Val Loss: 0.8977 | Val micro-F1: 0.2747
Epoch 03 | Train Loss: 0.8081 | Val Loss: 0.8895 | Val micro-F1: 0.2715
Epoch 04 | Train Loss: 0.7771 | Val Loss: 0.9035 | Val micro-F1: 0.2758
Epoch 05 | Train Loss: 0.7428 | Val Loss: 0.9217 | Val micro-F1: 0.2774
Epoch 06 | Train Loss: 0.7118 | Val Loss: 0.9401 | Val micro-F1: 0.2775
Epoch 07 | Train Loss: 0.6841 | Val Loss: 0.9758 | Val micro-F1: 0.2869
Epoch 08 | Train Loss: 0.6543 | Val Loss: 1.0058 | Val micro-F1: 0.2831
Epoch 09 | Train Loss: 0.6285 | Val Loss: 1.0424 | Val micro-F1: 0.2892


epoch,▁▂▃▃▄▅▆▆▇█
train_loss,█▆▅▄▄▃▃▂▁▁
val_loss,▂▁▁▁▂▂▃▅▆█
val_micro_f1,▁▄▅▄▅▆▆█▇█
epoch,9
train_loss,0.62849
val_loss,1.04244
val_micro_f1,0.28916


In [10]:
def get_average_word2vec(text, model, num_features=300):
    words = re.findall(r'\b\w+\b', text.lower())

    valid_vectors = [model[word] for word in words if word in model]

    if not valid_vectors:
        return np.zeros(num_features, dtype=np.float32)

    return np.mean(valid_vectors, axis=0)

w2v_google = gensim.downloader.load('word2vec-google-news-300')

print("Encoding Training Summaries...")
X_train_w2v = np.vstack([
    get_average_word2vec(text, w2v_google) for text in tqdm(X_train.astype(str))
])

print("Encoding Testing Summaries...")
X_test_w2v = np.vstack([
    get_average_word2vec(text, w2v_google) for text in tqdm(X_test.astype(str))
])

train_dataset_w2v = TextDataset(X_train_w2v, y_train)
test_dataset_w2v = TextDataset(X_test_w2v, y_test)

train_loader_ggle = DataLoader(train_dataset_w2v, batch_size=128, shuffle=True)
test_loader_ggle = DataLoader(test_dataset_w2v, batch_size=128, shuffle=False)

Encoding Training Summaries...


100%|██████████| 80000/80000 [00:28<00:00, 2817.56it/s]


Encoding Testing Summaries...


100%|██████████| 20000/20000 [00:05<00:00, 3333.39it/s]


In [11]:
model = MLP_basic(
            input_dim=300,
            num_classes=30
        ).to("cuda")
run_experiment(train_loader_ggle, test_loader_ggle, model, "MLP_ggle")

Running experiment on device: cuda


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: mateusz-katafiasz (models-uniwroc) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch 00 | Train Loss: 0.9234 | Val Loss: 0.8192 | Val micro-F1: 0.2919
Epoch 01 | Train Loss: 0.7761 | Val Loss: 0.7556 | Val micro-F1: 0.3241
Epoch 02 | Train Loss: 0.7351 | Val Loss: 0.7391 | Val micro-F1: 0.3309
Epoch 03 | Train Loss: 0.7114 | Val Loss: 0.7213 | Val micro-F1: 0.3361
Epoch 04 | Train Loss: 0.6927 | Val Loss: 0.7130 | Val micro-F1: 0.3339
Epoch 05 | Train Loss: 0.6779 | Val Loss: 0.7105 | Val micro-F1: 0.3519
Epoch 06 | Train Loss: 0.6660 | Val Loss: 0.7002 | Val micro-F1: 0.3389
Epoch 07 | Train Loss: 0.6551 | Val Loss: 0.6998 | Val micro-F1: 0.3505
Epoch 08 | Train Loss: 0.6455 | Val Loss: 0.6972 | Val micro-F1: 0.3487
Epoch 09 | Train Loss: 0.6354 | Val Loss: 0.6973 | Val micro-F1: 0.3492


epoch,▁▂▃▃▄▅▆▆▇█
train_loss,█▄▃▃▂▂▂▁▁▁
val_loss,█▄▃▂▂▂▁▁▁▁
val_micro_f1,▁▅▆▆▆█▆███
epoch,9
train_loss,0.63539
val_loss,0.69729
val_micro_f1,0.34918


In [12]:
model = ResNet_MLP().to("cuda")
run_experiment(train_loader_ggle, test_loader_ggle, model, "ResNet_ggle")

Running experiment on device: cuda


Epoch 00 | Train Loss: 0.8272 | Val Loss: 0.7406 | Val micro-F1: 0.3064
Epoch 01 | Train Loss: 0.7245 | Val Loss: 0.7120 | Val micro-F1: 0.3295
Epoch 02 | Train Loss: 0.6880 | Val Loss: 0.7077 | Val micro-F1: 0.3251
Epoch 03 | Train Loss: 0.6679 | Val Loss: 0.6982 | Val micro-F1: 0.3398
Epoch 04 | Train Loss: 0.6464 | Val Loss: 0.6902 | Val micro-F1: 0.3515
Epoch 05 | Train Loss: 0.6286 | Val Loss: 0.6944 | Val micro-F1: 0.3522
Epoch 06 | Train Loss: 0.6114 | Val Loss: 0.6919 | Val micro-F1: 0.3528
Epoch 07 | Train Loss: 0.5950 | Val Loss: 0.6961 | Val micro-F1: 0.3604
Epoch 08 | Train Loss: 0.5798 | Val Loss: 0.7011 | Val micro-F1: 0.3566
Epoch 09 | Train Loss: 0.5655 | Val Loss: 0.7181 | Val micro-F1: 0.3652


epoch,▁▂▃▃▄▅▆▆▇█
train_loss,█▅▄▄▃▃▂▂▁▁
val_loss,█▄▃▂▁▂▁▂▃▅
val_micro_f1,▁▄▃▅▆▆▇▇▇█
epoch,9
train_loss,0.56546
val_loss,0.71805
val_micro_f1,0.3652


In [18]:
def tokenize(text):
    return re.findall(r'\b\w+\b', str(text).lower())

sentences_train = [tokenize(text) for text in X_train]

custom_w2v = Word2Vec(
    sentences=sentences_train,
    vector_size=300,
    window=5,
    min_count=2,
    workers=4
)


Trenuję własny model Word2Vec (może to chwilę potrwać)...


KeyboardInterrupt: 

In [14]:
def get_average_word2vec(text, model, num_features=300):
    words = re.findall(r'\b\w+\b', text.lower())

    valid_vectors = [model[word] for word in words if word in model]

    if not valid_vectors:
        return np.zeros(num_features, dtype=np.float32)

    return np.mean(valid_vectors, axis=0)

print("Encoding Training Summaries...")
X_train_w2v = np.vstack([
    get_average_word2vec(text, custom_w2v.wv) for text in tqdm(X_train.astype(str))
])

print("Encoding Testing Summaries...")
X_test_w2v = np.vstack([
    get_average_word2vec(text, custom_w2v.wv) for text in tqdm(X_test.astype(str))
])

train_dataset_w2v = TextDataset(X_train_w2v, y_train)
test_dataset_w2v = TextDataset(X_test_w2v, y_test)

train_loader_custom = DataLoader(train_dataset_w2v, batch_size=128, shuffle=True)
test_loader_custom = DataLoader(test_dataset_w2v, batch_size=128, shuffle=False)

Encoding Training Summaries...


100%|██████████| 80000/80000 [00:30<00:00, 2656.39it/s]


Encoding Testing Summaries...


100%|██████████| 20000/20000 [00:07<00:00, 2506.50it/s]


In [16]:
model = MLP_basic(
            input_dim=300,
            num_classes=30
        ).to("cuda")
run_experiment(train_loader_custom, test_loader_custom, model, "MLP_custom")

Running experiment on device: cuda


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: mateusz-katafiasz (models-uniwroc) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch 00 | Train Loss: 0.9103 | Val Loss: 0.8295 | Val micro-F1: 0.2788
Epoch 01 | Train Loss: 0.8027 | Val Loss: 0.7908 | Val micro-F1: 0.2887
Epoch 02 | Train Loss: 0.7741 | Val Loss: 0.7919 | Val micro-F1: 0.3122
Epoch 03 | Train Loss: 0.7569 | Val Loss: 0.7905 | Val micro-F1: 0.3450
Epoch 04 | Train Loss: 0.7449 | Val Loss: 0.7600 | Val micro-F1: 0.3251
Epoch 05 | Train Loss: 0.7338 | Val Loss: 0.7545 | Val micro-F1: 0.3228
Epoch 06 | Train Loss: 0.7246 | Val Loss: 0.7592 | Val micro-F1: 0.3242
Epoch 07 | Train Loss: 0.7180 | Val Loss: 0.7619 | Val micro-F1: 0.3155
Epoch 08 | Train Loss: 0.7090 | Val Loss: 0.7500 | Val micro-F1: 0.3282
Epoch 09 | Train Loss: 0.7023 | Val Loss: 0.7445 | Val micro-F1: 0.3316


epoch,▁▂▃▃▄▅▆▆▇█
train_loss,█▄▃▃▂▂▂▂▁▁
val_loss,█▅▅▅▂▂▂▂▁▁
val_micro_f1,▁▂▅█▆▆▆▅▆▇
epoch,9
train_loss,0.7023
val_loss,0.74446
val_micro_f1,0.33159


In [17]:
model = ResNet_MLP().to("cuda")
run_experiment(train_loader_custom, test_loader_custom, model, "ResNet_custom")

Running experiment on device: cuda


Epoch 00 | Train Loss: 0.8708 | Val Loss: 0.8190 | Val micro-F1: 0.2907
Epoch 01 | Train Loss: 0.7835 | Val Loss: 0.7984 | Val micro-F1: 0.3141
Epoch 02 | Train Loss: 0.7543 | Val Loss: 0.7855 | Val micro-F1: 0.3056
Epoch 03 | Train Loss: 0.7374 | Val Loss: 0.7603 | Val micro-F1: 0.2999
Epoch 04 | Train Loss: 0.7216 | Val Loss: 0.7715 | Val micro-F1: 0.3125
Epoch 05 | Train Loss: 0.7111 | Val Loss: 0.7491 | Val micro-F1: 0.3244
Epoch 06 | Train Loss: 0.6995 | Val Loss: 0.7560 | Val micro-F1: 0.3129
Epoch 07 | Train Loss: 0.6901 | Val Loss: 0.7371 | Val micro-F1: 0.3336
Epoch 08 | Train Loss: 0.6818 | Val Loss: 0.7411 | Val micro-F1: 0.3319
Epoch 09 | Train Loss: 0.6727 | Val Loss: 0.7610 | Val micro-F1: 0.3149


epoch,▁▂▃▃▄▅▆▆▇█
train_loss,█▅▄▃▃▂▂▂▁▁
val_loss,█▆▅▃▄▂▃▁▁▃
val_micro_f1,▁▅▃▃▅▆▅██▅
epoch,9
train_loss,0.67271
val_loss,0.76101
val_micro_f1,0.31488


In [10]:
def tokenize(text):
    return re.findall(r'\b\w+\b', str(text).lower())

sentences_train = [tokenize(text) for text in X_train]

print("Trenuję własny model Word2Vec (może to chwilę potrwać)...")
custom_w2v_v2 = Word2Vec(
    sentences=sentences_train,
    vector_size=300,
    window=5,
    min_count=10,
    workers=4,
    epochs=15
)

Trenuję własny model Word2Vec (może to chwilę potrwać)...


In [11]:
def get_average_word2vec(text, model, num_features=300):
    words = re.findall(r'\b\w+\b', text.lower())

    valid_vectors = [model[word] for word in words if word in model]

    if not valid_vectors:
        return np.zeros(num_features, dtype=np.float32)

    return np.mean(valid_vectors, axis=0)

print("Encoding Training Summaries...")
X_train_w2v = np.vstack([
    get_average_word2vec(text, custom_w2v_v2.wv) for text in tqdm(X_train.astype(str))
])

print("Encoding Testing Summaries...")
X_test_w2v = np.vstack([
    get_average_word2vec(text, custom_w2v_v2.wv) for text in tqdm(X_test.astype(str))
])

train_dataset_w2v = TextDataset(X_train_w2v, y_train)
test_dataset_w2v = TextDataset(X_test_w2v, y_test)

train_loader_custom_v2 = DataLoader(train_dataset_w2v, batch_size=128, shuffle=True)
test_loader_custom_v2 = DataLoader(test_dataset_w2v, batch_size=128, shuffle=False)

Encoding Training Summaries...


100%|██████████| 80000/80000 [00:29<00:00, 2749.87it/s]


Encoding Testing Summaries...


100%|██████████| 20000/20000 [00:06<00:00, 3096.68it/s]


In [12]:
model = MLP_basic(
            input_dim=300,
            num_classes=30
        ).to("cuda")
run_experiment(train_loader_custom_v2, test_loader_custom_v2, model, "MLP_custom_v2")

Running experiment on device: cuda


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: mateusz-katafiasz (models-uniwroc) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch 00 | Train Loss: 0.8415 | Val Loss: 0.7844 | Val micro-F1: 0.3255
Epoch 01 | Train Loss: 0.7316 | Val Loss: 0.7269 | Val micro-F1: 0.3274
Epoch 02 | Train Loss: 0.6996 | Val Loss: 0.7153 | Val micro-F1: 0.3383
Epoch 03 | Train Loss: 0.6797 | Val Loss: 0.6981 | Val micro-F1: 0.3474
Epoch 04 | Train Loss: 0.6654 | Val Loss: 0.6943 | Val micro-F1: 0.3522
Epoch 05 | Train Loss: 0.6510 | Val Loss: 0.6943 | Val micro-F1: 0.3689
Epoch 06 | Train Loss: 0.6412 | Val Loss: 0.6892 | Val micro-F1: 0.3463
Epoch 07 | Train Loss: 0.6295 | Val Loss: 0.6931 | Val micro-F1: 0.3678
Epoch 08 | Train Loss: 0.6195 | Val Loss: 0.6995 | Val micro-F1: 0.3676
Epoch 09 | Train Loss: 0.6104 | Val Loss: 0.7046 | Val micro-F1: 0.3664


epoch,▁▂▃▃▄▅▆▆▇█
train_loss,█▅▄▃▃▂▂▂▁▁
val_loss,█▄▃▂▁▁▁▁▂▂
val_micro_f1,▁▁▃▅▅█▄███
epoch,9
train_loss,0.61044
val_loss,0.70465
val_micro_f1,0.36636


In [13]:
model = ResNet_MLP().to("cuda")
run_experiment(train_loader_custom_v2, test_loader_custom_v2, model, "ResNet_custom_v2")

Running experiment on device: cuda


Epoch 00 | Train Loss: 0.8234 | Val Loss: 0.7487 | Val micro-F1: 0.3106
Epoch 01 | Train Loss: 0.7273 | Val Loss: 0.7314 | Val micro-F1: 0.3153
Epoch 02 | Train Loss: 0.6962 | Val Loss: 0.7108 | Val micro-F1: 0.3335
Epoch 03 | Train Loss: 0.6751 | Val Loss: 0.7059 | Val micro-F1: 0.3337
Epoch 04 | Train Loss: 0.6606 | Val Loss: 0.7007 | Val micro-F1: 0.3280
Epoch 05 | Train Loss: 0.6457 | Val Loss: 0.6813 | Val micro-F1: 0.3565
Epoch 06 | Train Loss: 0.6333 | Val Loss: 0.6833 | Val micro-F1: 0.3549
Epoch 07 | Train Loss: 0.6210 | Val Loss: 0.6855 | Val micro-F1: 0.3494
Epoch 08 | Train Loss: 0.6088 | Val Loss: 0.7004 | Val micro-F1: 0.3503
Epoch 09 | Train Loss: 0.5993 | Val Loss: 0.6829 | Val micro-F1: 0.3520


epoch,▁▂▃▃▄▅▆▆▇█
train_loss,█▅▄▃▃▂▂▂▁▁
val_loss,█▆▄▄▃▁▁▁▃▁
val_micro_f1,▁▂▄▅▄██▇▇▇
epoch,9
train_loss,0.59933
val_loss,0.68288
val_micro_f1,0.35203


LSTM

In [8]:
import torch
import torch.nn as nn

class BiLSTM_Classifier(nn.Module):
    def __init__(self, embedding_layer=None, vocab_size=None, hidden_dim=128, num_classes=30, dropout_rate=0.5):
        super().__init__()

        if embedding_layer is not None:
            self.embedding = embedding_layer
        else:
            if vocab_size == None:
                raise ValueError("Musisz podać `embedding_layer` LUB `vocab_size`, aby zainicjalizować model!")

            self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=300, padding_idx=0)

        embedding_dim = self.embedding.embedding_dim

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(dropout_rate)
        self.classifier = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        x = x.to(self.embedding.weight.device, dtype=torch.long)
        embedded = self.embedding(x)

        out, _ = self.lstm(embedded)

        out = torch.max(out, dim=1)[0]

        out = self.dropout(out)
        logits = self.classifier(out)

        return logits

In [9]:
MAX_LEN = 100
MAX_VOCAB = 20000

def clean_and_tokenize(text):
    return re.findall(r'\b\w+\b', str(text).lower())

all_tokens = []
for text in tqdm(X_train):
    all_tokens.extend(clean_and_tokenize(text))

counts = Counter(all_tokens)
vocab = {word: i+2 for i, (word, _) in enumerate(counts.most_common(MAX_VOCAB-2))}
vocab["<PAD>"] = 0
vocab["<UNK>"] = 1

100%|██████████| 80000/80000 [00:07<00:00, 11284.84it/s]


In [10]:
class SequenceDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [11]:
del df

In [15]:
def text_to_indices(text, vocab, max_len):
    tokens = clean_and_tokenize(text)
    indices = [vocab.get(token, 1) for token in tokens]

    if len(indices) < max_len:
        indices = indices + [0] * (max_len - len(indices))
    else:
        indices = indices[:max_len]

    return np.array(indices, dtype=np.int64)

X_train_seq = np.vstack([text_to_indices(text, vocab, MAX_LEN) for text in tqdm(X_train)])
X_test_seq = np.vstack([text_to_indices(text, vocab, MAX_LEN) for text in tqdm(X_test)])

train_dataset_seq = SequenceDataset(X_train_seq, y_train)
test_dataset_seq = SequenceDataset(X_test_seq, y_test)

train_loader_lstm = DataLoader(train_dataset_seq, batch_size=128, shuffle=True)
test_loader_lstm = DataLoader(test_dataset_seq, batch_size=128, shuffle=False)

100%|██████████| 20000/20000 [00:01<00:00, 13674.80it/s]


In [12]:
def run_experiment(train_loader, test_loader, model, experiment_name):
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Przenosimy model na GPU przed konfiguracją czegokolwiek innego
    model = model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    y_train_np = np.asarray(y_train)
    pos_counts = y_train_np.sum(axis=0)
    neg_counts = len(y_train_np) - pos_counts
    pos_weight = torch.tensor(neg_counts / (pos_counts + 1e-6), dtype=torch.float32).to(device)

    # !!! KLUCZOWA POPRAWKA: Dodajemy .to(device) na końcu kryterium !!!
    criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight).to(device)

    wandb.init(
      project="Genre_Detector",
      name=experiment_name,
    )

    for epoch in range(10):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_f1 = evaluate(model, test_loader, criterion, device, threshold=0.5)

        print(f"Epoch {epoch:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val micro-F1: {val_f1:.4f}")

        wandb.log({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_micro_f1": val_f1
        })

    wandb.finish()

In [39]:
model_lstm = BiLSTM_Classifier(vocab_size=len(vocab))
run_experiment(train_loader_lstm, test_loader_lstm, model_lstm, 'Bi-LSTM_Scratch')

epoch,▁▂▃▄▅▆▇█
train_loss,█▅▄▃▃▂▁▁
val_loss,█▄▂▁▁▁▂▂
val_micro_f1,▁▃▄▆▆▇▇█
epoch,7
train_loss,0.61129
val_loss,0.82561
val_micro_f1,0.35239


Epoch 00 | Train Loss: 1.1069 | Val Loss: 0.9606 | Val micro-F1: 0.2365
Epoch 01 | Train Loss: 0.9121 | Val Loss: 0.8675 | Val micro-F1: 0.2846
Epoch 02 | Train Loss: 0.8301 | Val Loss: 0.8214 | Val micro-F1: 0.2957
Epoch 03 | Train Loss: 0.7653 | Val Loss: 0.8072 | Val micro-F1: 0.3141
Epoch 04 | Train Loss: 0.7163 | Val Loss: 0.7988 | Val micro-F1: 0.3224
Epoch 05 | Train Loss: 0.6797 | Val Loss: 0.8067 | Val micro-F1: 0.3413
Epoch 06 | Train Loss: 0.6430 | Val Loss: 0.8096 | Val micro-F1: 0.3423
Epoch 07 | Train Loss: 0.6098 | Val Loss: 0.8424 | Val micro-F1: 0.3529
Epoch 08 | Train Loss: 0.5840 | Val Loss: 0.8410 | Val micro-F1: 0.3474
Epoch 09 | Train Loss: 0.5653 | Val Loss: 0.8770 | Val micro-F1: 0.3663


epoch,▁▂▃▃▄▅▆▆▇█
train_loss,█▅▄▄▃▂▂▂▁▁
val_loss,█▄▂▁▁▁▁▃▃▄
val_micro_f1,▁▄▄▅▆▇▇▇▇█
epoch,9
train_loss,0.56526
val_loss,0.87704
val_micro_f1,0.36627


In [13]:
w2v_google = gensim.downloader.load('word2vec-google-news-300')

embedding_dim = 300
vocab_size = len(vocab)
embedding_matrix = np.zeros((vocab_size, embedding_dim), dtype=np.float32)

print("Przygotowuję macierz wag z Google News W2V...")
for word, idx in vocab.items():
    if word in w2v_google:
        embedding_matrix[idx] = w2v_google[word]
    else:
        embedding_matrix[idx] = np.random.normal(scale=0.6, size=(embedding_dim,))

embedding_matrix[0] = np.zeros(embedding_dim)

embedding_google_layer = nn.Embedding.from_pretrained(
    torch.FloatTensor(embedding_matrix),
    freeze=False,
    padding_idx=0
)

Przygotowuję macierz wag z Google News W2V...


In [16]:
model_lstm = BiLSTM_Classifier(embedding_layer=embedding_google_layer)
run_experiment(train_loader_lstm, test_loader_lstm, model_lstm, 'Bi-LSTM_ggle')

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: mateusz-katafiasz (models-uniwroc) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch 00 | Train Loss: 1.0221 | Val Loss: 0.8606 | Val micro-F1: 0.2758
Epoch 01 | Train Loss: 0.8000 | Val Loss: 0.7747 | Val micro-F1: 0.3084
Epoch 02 | Train Loss: 0.7017 | Val Loss: 0.7504 | Val micro-F1: 0.3378
Epoch 03 | Train Loss: 0.6309 | Val Loss: 0.7515 | Val micro-F1: 0.3504
Epoch 04 | Train Loss: 0.5725 | Val Loss: 0.7828 | Val micro-F1: 0.3625
Epoch 05 | Train Loss: 0.5248 | Val Loss: 0.8279 | Val micro-F1: 0.3705
Epoch 06 | Train Loss: 0.4791 | Val Loss: 0.8761 | Val micro-F1: 0.3807
Epoch 07 | Train Loss: 0.4433 | Val Loss: 0.9374 | Val micro-F1: 0.3872
Epoch 08 | Train Loss: 0.4114 | Val Loss: 1.0847 | Val micro-F1: 0.4113
Epoch 09 | Train Loss: 0.3847 | Val Loss: 1.1331 | Val micro-F1: 0.4088


epoch,▁▂▃▃▄▅▆▆▇█
train_loss,█▆▄▄▃▃▂▂▁▁
val_loss,▃▁▁▁▂▂▃▄▇█
val_micro_f1,▁▃▄▅▅▆▆▇██
epoch,9
train_loss,0.38475
val_loss,1.13313
val_micro_f1,0.4088


PACK PADDED SEQUENCE

In [17]:
class SequenceDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.float32)

        # Liczymy niezerowe elementy dla każdego wiersza, czyli rzeczywistą długość
        # clamp(min=1) chroni nas przed sytuacją, gdyby jakiś tekst był całkowicie pusty
        self.lengths = torch.clamp((self.X != 0).sum(dim=1), min=1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        # Dataset zwraca teraz 3 rzeczy: wejście, etykiety oraz długość tekstu
        return self.X[idx], self.y[idx], self.lengths[idx]

# Tworzymy loadery od nowa (kod z loaderami bez zmian, automatycznie ogarną nową paczkę danych)
train_dataset_seq = SequenceDataset(X_train_seq, y_train)
test_dataset_seq = SequenceDataset(X_test_seq, y_test)

train_loader_lstm = DataLoader(train_dataset_seq, batch_size=128, shuffle=True)
test_loader_lstm = DataLoader(test_dataset_seq, batch_size=128, shuffle=False)

In [18]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    total_samples = 0

    # Rozpakowujemy dodatkowo zmienną lengths
    for x, y, lengths in loader:
        x = x.to(device, dtype=torch.long)
        y = y.to(device, dtype=torch.float32)
        # lengths muszą zostać na CPU – PyTorch tego wymaga do pakowania sekwencji!

        optimizer.zero_grad()

        # Przekazujemy lengths bezpośrednio do modelu!
        logits = model(x, lengths)
        loss = criterion(logits, y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)
        total_samples += x.size(0)

    return total_loss / total_samples

@torch.no_grad()
def evaluate(model, loader, criterion, device, threshold=0.5):
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_targets = []

    for x, y, lengths in loader:
        x = x.to(device, dtype=torch.long)
        y = y.to(device, dtype=torch.float32)

        # Przekazujemy lengths bezpośrednio do modelu!
        logits = model(x, lengths)
        loss = criterion(logits, y)

        probs = torch.sigmoid(logits)
        preds = (probs > threshold).cpu().numpy()

        all_preds.append(preds)
        all_targets.append(y.cpu().numpy())

        total_loss += loss.item() * x.size(0)

    all_preds = np.vstack(all_preds)
    all_targets = np.vstack(all_targets)
    micro_f1 = f1_score(all_targets, all_preds, average="micro")

    return total_loss / len(loader.dataset), micro_f1

In [22]:
class BiLSTM_Classifier(nn.Module):
    def __init__(self, embedding_layer=None, vocab_size=None, hidden_dim=128, num_classes=30, dropout_rate=0.3):
        super().__init__()

        if embedding_layer is not None:
            self.embedding = embedding_layer
        else:
            if vocab_size is None:
                raise ValueError("Musisz podać `embedding_layer` LUB `vocab_size`!")
            self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=300, padding_idx=0)

        embedding_dim = self.embedding.embedding_dim

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(dropout_rate)
        self.classifier = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x, lengths):
        x = x.to(self.embedding.weight.device, dtype=torch.long)
        embedded = self.embedding(x)

        packed_embedded = nn.utils.rnn.pack_padded_sequence(
            embedded,
            lengths.to('cpu'),
            batch_first=True,
            enforce_sorted=False
        )

        packed_out, (hn, cn) = self.lstm(packed_embedded)
        out = torch.cat((hn[-2,:,:], hn[-1,:,:]), dim=1)

        out = self.dropout(out)
        logits = self.classifier(out)

        return logits

In [23]:
model_lstm_packed = BiLSTM_Classifier(vocab_size=len(vocab), dropout_rate=0.3)

run_experiment(train_loader_lstm, test_loader_lstm, model_lstm_packed, 'Bi-LSTM_PackPadded_Scratch')

Epoch 00 | Train Loss: 1.0980 | Val Loss: 0.9641 | Val micro-F1: 0.2307
Epoch 01 | Train Loss: 0.8806 | Val Loss: 0.8647 | Val micro-F1: 0.2757
Epoch 02 | Train Loss: 0.7614 | Val Loss: 0.8401 | Val micro-F1: 0.3031
Epoch 03 | Train Loss: 0.6688 | Val Loss: 0.8442 | Val micro-F1: 0.3200
Epoch 04 | Train Loss: 0.5897 | Val Loss: 0.8991 | Val micro-F1: 0.3460
Epoch 05 | Train Loss: 0.5226 | Val Loss: 0.9398 | Val micro-F1: 0.3596
Epoch 06 | Train Loss: 0.4675 | Val Loss: 1.0098 | Val micro-F1: 0.3739
Epoch 07 | Train Loss: 0.4189 | Val Loss: 1.0972 | Val micro-F1: 0.3824
Epoch 08 | Train Loss: 0.3809 | Val Loss: 1.1840 | Val micro-F1: 0.3924
Epoch 09 | Train Loss: 0.3491 | Val Loss: 1.3137 | Val micro-F1: 0.4018


epoch,▁▂▃▃▄▅▆▆▇█
train_loss,█▆▅▄▃▃▂▂▁▁
val_loss,▃▁▁▁▂▂▄▅▆█
val_micro_f1,▁▃▄▅▆▆▇▇██
epoch,9
train_loss,0.34908
val_loss,1.31373
val_micro_f1,0.40184


In [24]:
model_lstm = BiLSTM_Classifier(embedding_layer=embedding_google_layer, dropout_rate=0.3)
run_experiment(train_loader_lstm, test_loader_lstm, model_lstm, 'Bi-LSTM_PackPadded_ggle')

Epoch 00 | Train Loss: 0.8459 | Val Loss: 0.9061 | Val micro-F1: 0.3408
Epoch 01 | Train Loss: 0.4815 | Val Loss: 1.0492 | Val micro-F1: 0.3856
Epoch 02 | Train Loss: 0.3744 | Val Loss: 1.2459 | Val micro-F1: 0.4028
Epoch 03 | Train Loss: 0.3178 | Val Loss: 1.4385 | Val micro-F1: 0.4116
Epoch 04 | Train Loss: 0.2822 | Val Loss: 1.6601 | Val micro-F1: 0.4216
Epoch 05 | Train Loss: 0.2555 | Val Loss: 1.7770 | Val micro-F1: 0.4153
Epoch 06 | Train Loss: 0.2346 | Val Loss: 1.9982 | Val micro-F1: 0.4259
Epoch 07 | Train Loss: 0.2171 | Val Loss: 2.1992 | Val micro-F1: 0.4345
Epoch 08 | Train Loss: 0.2010 | Val Loss: 2.3331 | Val micro-F1: 0.4320
Epoch 09 | Train Loss: 0.1880 | Val Loss: 2.3377 | Val micro-F1: 0.4275


epoch,▁▂▃▃▄▅▆▆▇█
train_loss,█▄▃▂▂▂▁▁▁▁
val_loss,▁▂▃▄▅▅▆▇██
val_micro_f1,▁▄▆▆▇▇▇██▇
epoch,9
train_loss,0.18801
val_loss,2.33772
val_micro_f1,0.42753
